In [ ]:
# import


# globals

batch_size = 100
learning_rate = 0.003

In [ ]:
# dataset i dataloader

from torch.utils.data import Dataset, DataLoader
from torchvision.io import decode_image
from torchvision.datasets import ImageFolder

class CustomImageDataset(Dataset):
    def __init__(self, root):
        self.data = ImageFolder(root)

    def __len__(self):
    return len(self.data)


    def __getitem__(self, idx):
        image, label = self.data[idx]
 
        return image, label # image shape = 3x224x244


train_dataset = CustomImageDataset("data/train")
test_dataset = CustomImageDataset("data/test")

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# model

class CustomCNN_skinCancer(nn.Module):
    def __init__(self):
        super().__init__()

        self.seq = nn.Sequential(
            nn.Conv2d(in_channels = 3, out_channels = 48, kernel_size = (11,11), stride = 4), # (bs, 3, 224, 244) -> 219, 239 -> bs, 48, 55, 58
            nn.Conv2d(48, 128, (5,5)), # 53, 54
            nn.MaxPool2d(kernel_size = (2,2), stride = 2), # 27, 28
            nn.Conv2d(128, 192, (3,3)), # 26, 27
            nn.MaxPool2d((2,2)), # 13, 13
            nn.Conv2d(192, 192, (3,3)),# 12, 12
            nn.Conv2d(192, 128, (3,3)), # 11,11
            nn.MaxPool2d((2,2)), # 128,5,5

            nn.Flatten(),
            nn.Linear(3200, 2048),
            nn.ReLU(),
            nn.Linear(2048, 2)

            )

        
    def forward(self, X):
        return self.seq(X)

model = CustomCNN_skinCancer()
print(model)

In [ ]:
# train loop i test loop

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
# optimizer i loss fn
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# main

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")